## Notebook-Local Environment Checks and Installs

Use these cells only if you prefer installing from within Jupyter (venv already activated before launching). They verify CUDA and optionally install missing packages without leaving the notebook.

In [ ]:
# 0) Environment Sanity + Optional Installs
import sys, subprocess, pkgutil, shutil
print('Python:', sys.version)
try:
    import torch
    print('Torch:', torch.__version__, '| CUDA available:', torch.cuda.is_available())
except Exception as e:
    print('Torch not present or errored:', e)
    
def ensure(pkg, extras=None):
    name = pkg if not extras else f"{pkg}[{extras}]"
    if pkgutil.find_loader(pkg) is None:
        print('Installing', name)
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', name])
    else:
        print('Present:', name)
    
# Uncomment to install missing core packages from inside the notebook
# ensure('torch')  # Prefer installing via PowerShell with CUDA index URL
# ensure('transformers')
# ensure('accelerate')
# ensure('datasets')
# ensure('sentencepiece')
# ensure('safetensors')
# ensure('huggingface_hub', extras='hf_xet')

# Shadow GPU Benchmark & API Timing

Run these cells in your Shadow PC JupyterLab with the `shadow_ai` virtual environment activated.
Execute top-to-bottom. If any CUDA checks fail, revisit driver and PyTorch install.

## 1) Create and Activate Virtual Environment (PowerShell/Terminal)

Run this in PowerShell/Terminal (outside of Jupyter), then return here:

```powershell
py -3.11 -m venv .venv
.\.venv\Scripts\Activate
python -m pip install -U pip
```

## 2) Install JupyterLab and GPU PyTorch (PowerShell/Terminal)

Run these in your activated venv:

```powershell
pip install jupyterlab
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
pip install transformers accelerate datasets sentencepiece
```

## 3) Verify CUDA and PyTorch (PowerShell/Terminal)

Quick external check (optional):

```powershell
python -c "import torch; print('CUDA available:', torch.cuda.is_available()); print(torch.version.cuda, torch.__version__)"
```

In [ ]:
# 6) Simple GPU Inference (Transformers)
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
model_name = 'ProsusAI/finbert'
tok = AutoTokenizer.from_pretrained(model_name)
# Prefer safetensors; fall back to bin only if needed
mdl = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if torch.cuda.is_available() else None,
    low_cpu_mem_usage=True
)
mdl = mdl.to('cuda' if torch.cuda.is_available() else 'cpu')
batch = tok(["GPU test"]*8, return_tensors='pt', padding=True, truncation=True).to(mdl.device)
with torch.no_grad():
    out = mdl(**batch).logits
print('Logits shape:', out.shape, '| Device:', mdl.device)

In [ ]:
# 5) GPU Device Info
import torch
print('CUDA:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')
print('Device count:', torch.cuda.device_count())

## 4) Launch JupyterLab from VS Code Terminal (PowerShell/Terminal)

Start JupyterLab with no auth for local use (optional flags):

```powershell
jupyter lab --no-browser --NotebookApp.token='' --NotebookApp.password=''
```

In [ ]:
# 7) Mixed Precision and Batch Timing
import time, torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
name = 'ProsusAI/finbert'
texts = ["Market outlook remains positive."]*32
tok = AutoTokenizer.from_pretrained(name)
mdl = AutoModelForSequenceClassification.from_pretrained(name).to('cuda' if torch.cuda.is_available() else 'cpu')
batch = tok(texts, return_tensors='pt', padding=True, truncation=True).to(mdl.device)
torch.backends.cudnn.benchmark = mdl.device.type == 'cuda'
with torch.no_grad():
    _ = mdl(**batch)  # warmup
    t0 = time.time(); _ = mdl(**batch); t1 = time.time()
print('Batch time:', round(t1 - t0, 4), 's')
if mdl.device.type == 'cuda':
    from torch import autocast
    with torch.no_grad(), autocast(device_type='cuda', dtype=torch.float16):
        t0 = time.time(); _ = mdl(**batch); t1 = time.time()
    print('Mixed precision batch time:', round(t1 - t0, 4), 's')

In [ ]:
# 8) Troubleshooting Checks
import torch, platform
print('PyTorch:', torch.__version__)
print('CUDA build:', getattr(torch.version, 'cuda', 'cpu-only'))
print('GPU name:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no-gpu')
print('Python:', platform.python_version())
print('If CUDA False: install Studio driver, use --index-url cu121/cu126, and match Python 3.10–3.12.')

## Results Checklist

- Record GPU batch time and mixed precision time.
- Confirm `CUDA: True` and correct GPU name.
- If slower than expected, verify driver and use batch sizes 16–64.

Next: Run the pipeline script in your venv (`digest/digest_mvp.py`) and capture total timings for Round 5 analysis.